In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/benin.csv')
df.head()
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [ ]:
# Summary statistics for numeric columns
summary_stats = df.describe()
print(summary_stats)

# Missing values report
missing_values = df.isna().sum()
missing_percentage = (df.isna().sum() / len(df)) * 100
missing_report = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
print(missing_report[missing_report['Percentage'] > 5])  # Columns with >5% missing

In [ ]:
key_columns = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']
z_scores = df[key_columns].apply(zscore, nan_policy='omit')
outliers = (z_scores.abs() > 3).any(axis=1)
print(f"Number of outlier rows: {outliers.sum()}")

In [ ]:
for col in key_columns:
    df[col].fillna(df[col].median(), inplace=True)

In [ ]:
df.dropna(subset=key_columns, inplace=True)

In [ ]:
df_clean = df[~outliers]

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_clean['Timestamp'], df_clean['GHI'], label='GHI')
plt.plot(df_clean['Timestamp'], df_clean['DNI'], label='DNI')
plt.plot(df_clean['Timestamp'], df_clean['DHI'], label='DHI')
plt.plot(df_clean['Timestamp'], df_clean['Tamb'], label='Tamb')
plt.xlabel('Timestamp')
plt.ylabel('Values')
plt.title('Time Series of GHI, DNI, DHI, and Tamb')
plt.legend()
plt.show()

In [ ]:
df_clean['Month'] = df_clean['Timestamp'].dt.month
monthly_ghi = df_clean.groupby('Month')['GHI'].mean()
plt.figure(figsize=(8, 4))
monthly_ghi.plot(kind='bar')
plt.xlabel('Month')
plt.ylabel('Average GHI')
plt.title('Average GHI by Month')
plt.show()

In [3]:
cleaning_impact = df_clean.groupby('Cleaning')[['ModA', 'ModB']].mean()
cleaning_impact.plot(kind='bar', figsize=(8, 4))
plt.xlabel('Cleaning Status')
plt.ylabel('Average Sensor Reading')
plt.title('Average ModA and ModB by Cleaning Status')
plt.show()

NameError: name 'df_clean' is not defined

In [ ]:
corr_columns = ['GHI', 'DNI', 'DHI', 'TModA', 'TModB']
corr_matrix = df_clean[corr_columns].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df_clean['WS'], df_clean['GHI'], alpha=0.5)
plt.xlabel('Wind Speed (WS)')
plt.ylabel('GHI')
plt.title('WS vs. GHI')
plt.show()

In [ ]:
from windrose import WindroseAxes
ax = WindroseAxes.from_ax()
ax.bar(df_clean['WD'], df_clean['WS'], normed=True, opening=0.8, edgecolor='white')
ax.set_legend()
plt.title('Wind Rose Plot')
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(df_clean['GHI'], bins=30, edgecolor='black')
plt.xlabel('GHI')
plt.title('GHI Distribution')
plt.subplot(1, 2, 2)
plt.hist(df_clean['WS'], bins=30, edgecolor='black')
plt.xlabel('Wind Speed (WS)')
plt.title('WS Distribution')
plt.tight_layout()
plt.show()

In [ ]:
sns.lmplot(x='RH', y='Tamb', data=df_clean, height=6)
plt.title('RH vs. Tamb with Regression Line')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_clean['Tamb'], df_clean['GHI'], s=df_clean['RH']*10, alpha=0.5)
plt.xlabel('Ambient Temperature (Tamb)')
plt.ylabel('GHI')
plt.title('GHI vs. Tamb (Bubble Size = RH)')
plt.show()

In [ ]:
df_clean.to_csv('../data/benin_clean.csv', index=False)